# SWOT along-track SSH spectra in 2°×2° boxes — Dask parallel version

This notebook is a drop-in parallel replacement for **GSER_SSH_spectra.ipynb**.
All loader functions, helper functions, and plotting cells are identical; only
section 5 (the processing loop) is replaced by a Dask task graph.

### Why Dask over `ProcessPoolExecutor` here?
| | ProcessPoolExecutor | Dask |
|---|---|---|  
| Live progress | no | **Dashboard (port 8787)** |
| Memory pressure | no back-pressure | **worker memory limits** |
| Scale-out | local only | local → SLURM/PBS/K8s with same code |
| Task graph inspection | no | **`dask.visualize()`** |
| Retry on worker crash | no | yes |

### Strategy
Dask is used **only for the embarrassingly-parallel file-level work** (load + compute spectra per granule). The aggregation into `xarray.Dataset` objects and all plotting happen in the main process on plain numpy/xarray, as before — this avoids the overhead of trying to Dask-ify the `swot_analysis` internals.

Each Dask task processes one file and returns a compact list of `SegmentSpectrum` dataclasses (the same picklable objects as before). Because results come back as tiny numpy arrays — not the 30 MB raw granule — network/IPC transfer is negligible.

## 0. Imports

In [1]:
import glob
import os
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe

# ── Dask ─────────────────────────────────────────────────────────────────────
import dask
import dask.bag as db
from dask.distributed import Client, LocalCluster, progress, wait

from swot_analysis import compute_pass_spectra

## 1. Configuration
Identical to GSER_SSH_spectra.ipynb — change only these values.

In [2]:
# ── Input files ───────────────────────────────────────────────────────────────
DATA_DIR    = '/Users/zoecas/Documents/data/'
DATA_SUBDIR = 'SWOT_L2_LR_SSH_D_D-20260702_095005/'
FILE_PATTERN = 'SWOT_L2_LR_SSH_Expert_*.nc'

# ── Region of interest ────────────────────────────────────────────────────────
LAT_MIN, LAT_MAX = 25.0, 55.0
LON_MIN, LON_MAX = 280.0, 330.0

# ── Spectral parameters ───────────────────────────────────────────────────────
SEGMENT_LENGTH_KM = 250.0
OVERLAP           = 0.5
MAX_GAP_FRACTION  = 0.25

# ── Box size ──────────────────────────────────────────────────────────────────
BOX_DEG = 2.0

# ── Expert-product variable names ─────────────────────────────────────────────
SSH_VAR = 'ssha_karin_2'
HRET    = True   # add internal tide correction (height_cor_xover + internal_tide_hret)

# ── Dask cluster parameters ───────────────────────────────────────────────────
# N_WORKERS : number of worker processes.  Rule of thumb: #physical cores - 1.
# THREADS_PER_WORKER : keep at 1 (GIL + HDF5 thread-safety; see note in cell below).
# MEMORY_LIMIT : per-worker RAM cap (string: '4GB', '8GB', …).
#                Dask will spill to disk if exceeded — but spilling is slow;
#                see the scaling analysis in section 6 for how to size this.
N_WORKERS          = 4
THREADS_PER_WORKER = 1       # do NOT increase
MEMORY_LIMIT       = '4GB'   # per worker

## 2. Start the Dask cluster and open the Dashboard

**`LocalCluster`** spawns `N_WORKERS` independent Python processes on the same machine.

> ⚠️ `threads_per_worker=1` is intentional:
> the `swot_analysis` spectral loops are GIL-bound, and the underlying
> HDF5/netCDF4 library is not thread-safe for concurrent reads from the same
> process. Each worker uses one thread, and processes are isolated from one
> another, so there is no contention on either count.

Once the cell runs, click the **Dashboard link** printed below — or open
`http://localhost:8787` in your browser — to watch the task stream, worker
memory and CPU live while the processing loop runs in section 5.

In [3]:
cluster = LocalCluster(
    n_workers=N_WORKERS,
    threads_per_worker=THREADS_PER_WORKER,
    memory_limit=MEMORY_LIMIT,
    dashboard_address=':8787',
)
client = Client(cluster)
print(f"Dashboard: {client.dashboard_link}")
client   # displays the interactive widget in JupyterLab

Dashboard: http://127.0.0.1:8787/status


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 4,Total memory: 14.90 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:52289,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:52302,Total threads: 1
Dashboard: http://127.0.0.1:52308/status,Memory: 3.73 GiB
Nanny: tcp://127.0.0.1:52292,


## 3. Helper functions
Identical to GSER_SSH_spectra.ipynb (loader, `segments_to_dataset`, `build_box_dataset`).

In [4]:
def load_swot_l2_expert(filepath: str, ssh_var: str = 'ssha_karin_2',
                         HRET: bool = True) -> dict:
    """
    Expert-product loader — identical to GSER_SSH_spectra.ipynb.
    Runs inside each Dask worker process; all imports are re-imported
    locally to be safe under both fork and spawn start methods.
    """
    import numpy as np
    import xarray as xr

    ds = xr.open_dataset(filepath)
    if HRET:
        ssha = (ds[ssh_var] + ds['height_cor_xover']
                + ds['internal_tide_hret']).values.astype(float)
    else:
        ssha = (ds[ssh_var] + ds['height_cor_xover']).values.astype(float)

    for qual_name in (f'{ssh_var}_qual', 'ssha_karin_2_qual', 'ssh_karin_2_qual'):
        if qual_name in ds.variables:
            ssha = np.where(ds[qual_name].values == 0, ssha, np.nan)
            break

    for scf_name in ('ancillary_surface_classification_flag',
                      'surface_classification_flag'):
        if scf_name in ds.variables:
            ssha = np.where(ds[scf_name].values == 0, ssha, np.nan)
            break

    if 'surface_type' in ds.variables:
        ssha = np.where(ds['surface_type'].values == 0, ssha, np.nan)

    lat = ds['latitude'].values.astype(float)
    lon = ds['longitude'].values.astype(float)
    xt  = ds['cross_track_distance'].values.astype(float)
    ds.close()
    return {'ssha': ssha, 'latitude': lat, 'longitude': lon,
            'cross_track_distance': xt}


def segments_to_dataset(all_segments: list, wavenumber: np.ndarray) -> xr.Dataset:
    """Pack a flat list of SegmentSpectrum objects into an xarray.Dataset."""
    if not all_segments:
        raise ValueError('No segments to pack.')
    psd = np.stack([s.psd for s in all_segments], axis=0)
    return xr.Dataset(
        data_vars=dict(
            psd=(['segment', 'wavenumber'], psd,
                 {'long_name': 'Along-track SSH(A) PSD',
                  'units': 'm^2 / (cycles/km)'}),
            lat_mean=(['segment'], [s.lat_mean for s in all_segments],
                      {'units': 'degrees_north'}),
            lat_min=(['segment'],  [s.lat_min  for s in all_segments],
                     {'units': 'degrees_north'}),
            lat_max=(['segment'],  [s.lat_max  for s in all_segments],
                     {'units': 'degrees_north'}),
            lon_mean=(['segment'], [s.lon_mean for s in all_segments],
                      {'units': 'degrees_east'}),
            n_pixels_used=(['segment'], [s.n_pixels_used  for s in all_segments]),
            valid_fraction=(['segment'], [s.valid_fraction for s in all_segments]),
            swath=(['segment'],   [s.swath    for s in all_segments]),
            granule=(['segment'], [s._granule for s in all_segments]),
        ),
        coords=dict(
            wavenumber=('wavenumber', wavenumber, {'units': 'cycles/km'}),
            segment=('segment', np.arange(len(all_segments))),
        ),
    )


def build_box_dataset(ds_segments: xr.Dataset, box_deg: float = 2.0) -> xr.Dataset:
    """Bin segments into box_deg×box_deg boxes and compute mean PSD per box."""
    lat = ds_segments['lat_mean'].values
    lon = ds_segments['lon_mean'].values
    lat_idx = np.floor(lat / box_deg).astype(int)
    lon_idx = np.floor(lon / box_deg).astype(int)

    box_to_segs = defaultdict(list)
    for i, key in enumerate(zip(lat_idx, lon_idx)):
        box_to_segs[key].append(i)

    psd_all    = ds_segments['psd'].values
    wavenumber = ds_segments['wavenumber'].values
    rows_lat, rows_lon, rows_n, rows_psd = [], [], [], []
    li_out, loi_out = [], []

    for (li, loi), idxs in sorted(box_to_segs.items()):
        rows_psd.append(np.nanmean(psd_all[idxs, :], axis=0))
        rows_n.append(len(idxs))
        rows_lat.append((li  + 0.5) * box_deg)
        rows_lon.append((loi + 0.5) * box_deg)
        li_out.append(li)
        loi_out.append(loi)

    return xr.Dataset(
        data_vars=dict(
            mean_psd=(['box', 'wavenumber'], np.stack(rows_psd, axis=0),
                      {'long_name': 'Box-mean SSH(A) PSD',
                       'units': 'm^2 / (cycles/km)'}),
            n_segments=(['box'], rows_n,
                        {'long_name': 'Number of segments in box'}),
            lat_center=(['box'], rows_lat,
                        {'long_name': 'Box centre latitude',
                         'units': 'degrees_north'}),
            lon_center=(['box'], rows_lon,
                        {'long_name': 'Box centre longitude',
                         'units': 'degrees_east'}),
            lat_idx=(['box'], li_out),
            lon_idx=(['box'], loi_out),
        ),
        coords=dict(
            wavenumber=('wavenumber', wavenumber, {'units': 'cycles/km'}),
            box=('box', np.arange(len(rows_n))),
        ),
        attrs=dict(
            box_size_deg=box_deg,
            description=f'Mean SSH(A) PSD in {box_deg}°×{box_deg}° boxes.',
        ),
    )

## 4. Worker function

This is the unit of work dispatched to each Dask worker. It is a plain Python
function (no Dask internals); Dask simply calls it in a worker process and
serialises the return value back to the scheduler. The return value is a small
list of `SegmentSpectrum` dataclasses — not the 30 MB raw granule array —
so inter-process transfer is fast regardless of how many workers are used.

Errors are caught and returned as `(filepath, error_string, None)` so one bad
granule never cancels the whole computation.

In [5]:
def process_one_file(filepath, ssh_var, hret,
                      segment_length_km, overlap, max_gap_fraction,
                      lat_min, lat_max, lon_min, lon_max):
    """
    Dask worker task: load one Expert granule, compute spectra, filter to
    region of interest, return (granule_stem, list[SegmentSpectrum], wavenumber).
    On any error returns (filepath, error_string, None) with a full traceback
    so the cause is immediately visible in the gather cell.
    """
    import traceback as _tb
    from swot_analysis import compute_pass_spectra
    from pathlib import Path

    try:
        data = load_swot_l2_expert(filepath, ssh_var=ssh_var, HRET=hret)
    except Exception:
        return (filepath, f'load error:\n{_tb.format_exc()}', None)

    try:
        result = compute_pass_spectra(
            ssha=data['ssha'],
            latitude=data['latitude'],
            longitude=data['longitude'],
            cross_track_distance=data['cross_track_distance'],
            segment_length_km=segment_length_km,
            overlap=overlap,
            max_gap_fraction=max_gap_fraction,
        )
    except Exception:
        return (filepath, f'spectra error:\n{_tb.format_exc()}', None)

    granule_name = Path(filepath).stem
    kept, wavenumber = [], None
    for res in result.values():
        if res.n_segments_used == 0:
            continue
        if wavenumber is None:
            wavenumber = res.wavenumber
        for seg in res.segments:
            if (lat_min <= seg.lat_mean <= lat_max and
                    lon_min <= seg.lon_mean <= lon_max):
                seg._granule = granule_name
                kept.append(seg)

    return (granule_name, kept, wavenumber)

## 5. Parallel processing with Dask

### How it works

```
filepaths  ──►  client.map(process_one_file, ...)  ──►  futures[]
                    │ dispatched to N_WORKERS processes
                    │ visible in Dashboard → Task Stream
                    ▼
             progress(futures)   ← live bar in notebook
             wait(futures)       ← block until all done
                    ▼
             gather results  ──►  flat list of SegmentSpectrum
```

### Dashboard tabs to watch
| Tab | What to watch |
|---|---|
| **Task Stream** | colour-coded bars: orange=load, blue=spectra; gaps = scheduler overhead |
| **Workers** | per-worker CPU (aim for ≥ 80%) and memory (watch for memory pressure) |
| **Progress** | overall fraction done |
| **Bytes stored** | amount of intermediate data held in worker memory — should stay low since results are small SegmentSpectrum objects |

> Open `http://localhost:8787` in a browser tab **before** running this cell.

In [6]:
filepaths = sorted(glob.glob(os.path.join(DATA_DIR, DATA_SUBDIR, FILE_PATTERN)))
print(f'{len(filepaths)} Expert granule files found.')

# Fixed kwargs forwarded to every worker call
worker_kw = dict(
    ssh_var=SSH_VAR, hret=HRET,
    segment_length_km=SEGMENT_LENGTH_KM, overlap=OVERLAP,
    max_gap_fraction=MAX_GAP_FRACTION,
    lat_min=LAT_MIN, lat_max=LAT_MAX,
    lon_min=LON_MIN, lon_max=LON_MAX,
)

# Submit all tasks at once — Dask queues them and feeds workers as they
# become free.  client.map() returns immediately with a list of Future objects.
futures = client.map(process_one_file, filepaths,
                      **{k: [v] * len(filepaths) for k, v in worker_kw.items()})

# Live progress bar inside the notebook (also visible on the Dashboard)
progress(futures)

# Block until all futures are complete
wait(futures)

152 Expert granule files found.


DoneAndNotDoneFutures(done={<Future: finished, type: tuple, key: process_one_file-7b9319c2e3f2af98071a868c636d55dd>, <Future: finished, type: tuple, key: process_one_file-04f7e764070a1dc04b698c19a68311b8>, <Future: finished, type: tuple, key: process_one_file-d0c26f4496ee5d7af54238b23aa8ef3b>, <Future: finished, type: tuple, key: process_one_file-992c0e3061e762c192001cc2c9c55adc>, <Future: finished, type: tuple, key: process_one_file-41ac05252dd86a93468c54ca2cd87368>, <Future: finished, type: tuple, key: process_one_file-a10c393b43165ac7f548f14eab8ca625>, <Future: finished, type: tuple, key: process_one_file-1e45e9199f78537bafb89503b2124e0f>, <Future: finished, type: tuple, key: process_one_file-75483d88b750efd05a34d59a2108afa6>, <Future: finished, type: tuple, key: process_one_file-336bb5f30fc49a2822b57e9f68634579>, <Future: finished, type: tuple, key: process_one_file-61434b0fb01142f0e01a806f5a36b50e>, <Future: finished, type: tuple, key: process_one_file-45b8070adb167f40d321406dcd6f

In [7]:
# Gather results back to the main process
raw_results = client.gather(futures)

all_segments, wavenumber, skipped = [], None, []

for item in raw_results:
    name, payload, wk = item
    if isinstance(payload, str):                 # error string
        skipped.append((name, payload))
    else:
        all_segments.extend(payload)
        if wavenumber is None and wk is not None:
            wavenumber = wk

print(f'\nDone. {len(all_segments)} segments retained in region.')
print(f'{len(skipped)} file(s) skipped.')
if skipped:
    for name, err in skipped:
        print(f'  {name}: {err}')


Done. 0 segments retained in region.
152 file(s) skipped.
  /Users/zoecas/Documents/data/SWOT_L2_LR_SSH_D_D-20260702_095005/SWOT_L2_LR_SSH_Expert_014_380_20240501T001814_20240501T010942_PGD0_01.nc: load error:
Traceback (most recent call last):
  File "/var/folders/fv/pmf052vs1g77j8b23nzvlkmm0000gr/T/ipykernel_64562/997132464.py", line 15, in process_one_file
  File "/var/folders/fv/pmf052vs1g77j8b23nzvlkmm0000gr/T/ipykernel_64562/1818684588.py", line 14, in load_swot_l2_expert
AttributeError: 'function' object has no attribute 'astype'

  /Users/zoecas/Documents/data/SWOT_L2_LR_SSH_D_D-20260702_095005/SWOT_L2_LR_SSH_Expert_014_382_20240501T020108_20240501T025153_PGD0_01.nc: load error:
Traceback (most recent call last):
  File "/var/folders/fv/pmf052vs1g77j8b23nzvlkmm0000gr/T/ipykernel_64562/997132464.py", line 15, in process_one_file
  File "/var/folders/fv/pmf052vs1g77j8b23nzvlkmm0000gr/T/ipykernel_64562/1818684588.py", line 14, in load_swot_l2_expert
AttributeError: 'function' obj

### Dashboard interpretation guide

Run this cell **after** processing to print a structured summary that mirrors
what the Dashboard shows, so it is reproducible in outputs/CI.

In [ ]:
info = client.scheduler_info()
workers = info['workers']
print(f"{'Worker':<45} {'CPU%':>6} {'Mem used':>10} {'Tasks done':>12}")
print('-' * 80)
for addr, w in workers.items():
    mem_mb  = w['metrics']['memory'] / 1024**2
    cpu_pct = w['metrics']['cpu']
    ntasks  = w['metrics']['executing'] + w['metrics'].get('task_counts', {}).get('memory', 0)
    print(f"{addr:<45} {cpu_pct:>6.1f} {mem_mb:>9.0f}M {ntasks:>12}")

## 6. Build segment and box datasets
Identical to GSER_SSH_spectra.ipynb from here onward.

In [ ]:
if not all_segments:
    raise RuntimeError('No segments retained — check region bounds, '
                        'file pattern, and SSH variable name.')

ds_segments = segments_to_dataset(all_segments, wavenumber)
print(f'Segment dataset: {ds_segments.sizes}')

ds_boxes = build_box_dataset(ds_segments, box_deg=BOX_DEG)
print(f'Box dataset: {ds_boxes.sizes["box"]} boxes with at least 1 segment')
ds_boxes

## 7. Local-computing scaling analysis

At what point does local computation become impractical?
The two binding constraints are **disk I/O** and **RAM**.
Run this cell to estimate both for your specific machine and dataset.

In [ ]:
import psutil

# ── Machine parameters (auto-detected) ───────────────────────────────────────
ram_total_gb   = psutil.virtual_memory().total / 1024**3
n_cpus_logical = psutil.cpu_count(logical=True)
n_cpus_phys    = psutil.cpu_count(logical=False)

# ── File parameters ───────────────────────────────────────────────────────────
FILE_SIZE_MB   = 30.0     # per Expert granule
# One granule loaded into RAM: raw arrays (ssha + lat + lon + xtrack).
# ssha_karin_2 for Expert: (num_lines=9866, num_pixels=69), float32 → ~2.6 MB
# After casting to float64 + masking: ~5.2 MB active per granule in a worker
RAM_PER_WORKER_ACTIVE_MB = FILE_SIZE_MB * 0.18   # ~18% of file size stays hot

# ── Derived estimates ─────────────────────────────────────────────────────────
# Safe RAM: leave 20% for the OS + scheduler + notebook kernel
safe_ram_gb    = ram_total_gb * 0.80
max_workers_ram = int(safe_ram_gb * 1024 / RAM_PER_WORKER_ACTIVE_MB)

# Max files that fit comfortably in RAM if results are kept in memory
# (ds_segments PSD array: N_seg × N_k × 8 bytes; ~50 segs/file × 63 k-bins × 8 B ≈ 25 kB/file)
SEGS_PER_FILE  = len(all_segments) / max(len(filepaths), 1)
N_K            = len(wavenumber) if wavenumber is not None else 63
results_mb_per_file = SEGS_PER_FILE * N_K * 8 / 1024**2
max_files_ram  = int(safe_ram_gb * 1024 / (RAM_PER_WORKER_ACTIVE_MB + results_mb_per_file))

# Throughput estimate: ~8–15 s per file on a modern laptop (serial);
# with N_WORKERS it parallelises well (I/O bound, not CPU bound for large files)
T_FILE_S       = 10.0  # seconds per file, single-threaded estimate
T_parallel_min = lambda n: (n * T_FILE_S) / (60 * N_WORKERS)

print('=' * 60)
print(f'Machine:            {ram_total_gb:.1f} GB RAM, '
      f'{n_cpus_phys} physical / {n_cpus_logical} logical CPUs')
print(f'Dask workers:       {N_WORKERS}  (threads_per_worker=1)')
print(f'Safe RAM budget:    {safe_ram_gb:.1f} GB  (80% of total)')
print()
print(f'Granule file size:  {FILE_SIZE_MB:.0f} MB')
print(f'RAM hot per worker: ~{RAM_PER_WORKER_ACTIVE_MB:.1f} MB  (active arrays)')
print(f'Results per file:   ~{results_mb_per_file*1024:.0f} kB  '
      f'({SEGS_PER_FILE:.1f} segs × {N_K} k-bins × 8 B)')
print()
print(f'Max simultaneous workers (RAM):  {max_workers_ram}')
print(f'Max files before results alone')
print(f'  overflow safe RAM:             ~{max_files_ram:,} files  '
      f'({max_files_ram * FILE_SIZE_MB / 1024:.0f} GB of raw data)')
print()
print('Estimated wall-clock time:')
for n in (100, 500, 1000, 5000, 10000):
    print(f'  {n:>6} files → {T_parallel_min(n):>7.1f} min '
          f'  ({n * FILE_SIZE_MB / 1024:.0f} GB raw)')
print()
print('─' * 60)
print('LOCAL COMPUTING LIMITS (rough rules of thumb):')
print()
print(f'  ✅  < ~{max_files_ram:,} files / {max_files_ram * FILE_SIZE_MB/1024:.0f} GB raw:')
print( '       Comfortable on this machine with current settings.')
print()
print(f'  ⚠️   ~{max_files_ram:,} – {max_files_ram*3:,} files:')
print( '       Results still fit in RAM but Dask will spill worker state')
print( '       to disk. Set MEMORY_LIMIT lower and use ds_segments.to_netcdf()')
print( '       to checkpoint after each batch of files.')
print()
print(f'  ❌  > ~{max_files_ram*3:,} files / {max_files_ram*3*FILE_SIZE_MB/1024:.0f} GB raw:')
print( '       Exceeds safe local RAM even with spilling.')
print( '       Recommended: switch to a Dask cluster (SLURM/PBS via')
print( '       dask_jobqueue) or cloud workers (Coiled, Dask Gateway).')
print( '       Code change required: replace LocalCluster with e.g.:')
print( '         from dask_jobqueue import SLURMCluster')
print( '         cluster = SLURMCluster(cores=1, memory="8GB", ...) ')
print( '         cluster.scale(jobs=50)')
print( '       The rest of this notebook runs unchanged.')
print('=' * 60)

## 8. Diagnostics — identical to GSER_SSH_spectra.ipynb

In [ ]:
print('Segments per swath:')
for sw in ('left', 'right'):
    n = int((ds_segments['swath'] == sw).sum())
    print(f'  {sw}: {n}')

unique_granules = len(np.unique(ds_segments['granule'].values))
print(f'Unique granules contributing segments: {unique_granules}')
print(f'Boxes with ≥  5 segments: {int((ds_boxes["n_segments"] >=  5).sum())}')
print(f'Boxes with ≥ 10 segments: {int((ds_boxes["n_segments"] >= 10).sum())}')
print(f'Max segments in a box:    {int(ds_boxes["n_segments"].max())}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(
    ds_boxes['lon_center'].values, ds_boxes['lat_center'].values,
    c=ds_boxes['n_segments'].values, s=(BOX_DEG * 60) ** 1.5,
    marker='s', cmap='YlOrRd',
    norm=mcolors.LogNorm(vmin=1, vmax=float(ds_boxes['n_segments'].max())),
    edgecolors='none',
)
plt.colorbar(sc, ax=ax, label='Number of segments')
ax.set_xlim(LON_MIN, LON_MAX); ax.set_ylim(LAT_MIN, LAT_MAX)
ax.set_xlabel('Longitude [°E]'); ax.set_ylabel('Latitude [°N]')
ax.set_title(f'Segment count per {BOX_DEG}°×{BOX_DEG}° box')
ax.grid(True, alpha=0.3); fig.tight_layout()

In [ ]:
K_MIN, K_MAX = 1/500, 1/50
k      = ds_boxes['wavenumber'].values
k_mask = (k >= K_MIN) & (k <= K_MAX)
dk     = np.gradient(k)
energy = np.nansum(ds_boxes['mean_psd'].values[:, k_mask] * dk[k_mask], axis=1)

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(
    ds_boxes['lon_center'].values, ds_boxes['lat_center'].values,
    c=energy, s=(BOX_DEG * 60) ** 1.5, marker='s', cmap='plasma',
    norm=mcolors.LogNorm(vmin=np.nanpercentile(energy, 5),
                          vmax=np.nanpercentile(energy, 95)),
    edgecolors='none',
)
plt.colorbar(sc, ax=ax,
             label=f'Integrated PSD  [m²]  ({1/K_MAX:.0f}–{1/K_MIN:.0f} km)')
ax.set_xlim(LON_MIN, LON_MAX); ax.set_ylim(LAT_MIN, LAT_MAX)
ax.set_xlabel('Longitude [°E]'); ax.set_ylabel('Latitude [°N]')
ax.set_title('Integrated spectral energy per box')
ax.grid(True, alpha=0.3); fig.tight_layout()

In [ ]:
MIN_SEGMENTS_PER_BOX = 5
good = ds_boxes.where(ds_boxes['n_segments'] >= MIN_SEGMENTS_PER_BOX, drop=True)
lat_vals = good['lat_center'].values
norm = plt.Normalize(vmin=lat_vals.min(), vmax=lat_vals.max())
cmap = plt.get_cmap('viridis')

fig, ax = plt.subplots(figsize=(8, 5))
for i in range(good.sizes['box']):
    box = good.isel(box=i)
    ax.loglog(box['wavenumber'], box['mean_psd'],
              color=cmap(norm(float(box['lat_center']))), lw=1.5, alpha=0.85)

k_ref = good['wavenumber'].values[2:]
psd0  = float(good['mean_psd'].isel(box=0).values[2])
k0    = k_ref[0]
for exp, ls in ((-2, '--'), (-11/3, ':'), (-5, '-.')):  
    lbl = {-2: '$k^{-2}$', -11/3: '$k^{-11/3}$', -5: '$k^{-5}$'}[exp]
    ax.loglog(k_ref, psd0 * (k_ref / k0) ** exp, ls, color='gray', label=lbl)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
plt.colorbar(sm, ax=ax, label='Box centre latitude [°N]')
ax.set_xlabel('Wavenumber [cycles/km]')
ax.set_ylabel('PSD [m² / (cycles/km)]')
ax.set_title(f'Mean SSHA spectra per {BOX_DEG}°×{BOX_DEG}° box  '
             f'(≥{MIN_SEGMENTS_PER_BOX} segs, Dask parallel run)')
ax.legend(); ax.grid(True, which='both', alpha=0.3); fig.tight_layout()

## 9. Save and shut down

In [ ]:
# ds_segments.to_netcdf('swot_expert_segments_dask.nc')
# ds_boxes.to_netcdf('swot_expert_box_spectra_dask.nc')

In [8]:
# Gracefully shut down the cluster when done.
# Skip this if you want to re-use the cluster for another run.
client.close()
cluster.close()